# Home Credit Default Risk — XGBoost Model

**Bonus task:** train a model to predict loan default (`TARGET`) from the engineered feature table.

This notebook reads the feature table produced by the Spark pipeline (`feature_engineering.py`) from HDFS, prepares it for modeling, trains an XGBoost classifier, and evaluates it with ROC-AUC — the metric the original Kaggle competition uses and the appropriate choice for an imbalanced binary target.

**Pipeline context:** the heavy distributed work (cleaning, joining `application_train` with aggregated `bureau`, building features) was already done in Spark and written to HDFS as a Hudi table. This notebook only does the modeling step, reading that prepared table.

## 1. Read the feature table from HDFS via PySpark

Read the Hudi feature table previously created, through Spark, then convert to a pandas DataFrame. The dataset is ~307k rows, which fits comfortably in memory for single-machine XGBoost training.

In [ ]:
import findspark
findspark.init("/opt/spark")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("XGBoostDataLoad")
    .config("spark.hadoop.fs.defaultFS", "hdfs://hdfs-namenode:9000")
    .getOrCreate()
)

spark_df = spark.read.format("hudi").load("hdfs://hdfs-namenode:9000/project/features")

# Drop Hudi's internal metadata columns so they don't leak into the features
hoodie_cols = [c for c in spark_df.columns if c.startswith("_hoodie_")]
spark_df = spark_df.drop(*hoodie_cols)

print(f"Rows: {spark_df.count()}, Columns: {len(spark_df.columns)}")

In [ ]:
# Convert to pandas for modeling
df = spark_df.toPandas()
spark.stop()
df.shape

## 2. Separate the target and drop identifier columns

`TARGET` is what we predict (1 = defaulted, 0 = repaid). `SK_ID_CURR` is just an applicant ID with no predictive value, so it's dropped from the features.

In [ ]:
y = df["TARGET"]
X = df.drop(columns=["TARGET", "SK_ID_CURR"])

print(f"Feature matrix: {X.shape}")
print(f"Class balance:\n{y.value_counts()}")
print(f"Default rate: {y.mean():.1%}")

## 3. Encode categorical columns

XGBoost needs numeric input. We label-encode the string columns (convert each category to an integer).

Since XGBoost is tree-based, we use label encoding so it is robust to the artificial ordinality that label encoding introduces, a tree can isolate any single category through successive splits. One-hot encoding would be necessary for a linear model, but here it would explode high-cardinality columns (e.g. `ORGANIZATION_TYPE` has ~58 values) into hundreds of columns for no accuracy gain on a tree ensemble.

String nulls were already filled with `"Unknown"` in the Spark sanitize step, so they simply become one more category.

In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
print(f"Encoding {len(categorical_cols)} categorical columns")

for col in categorical_cols:
    le = LabelEncoder()
    # Cast to str so any residual nulls encode as a category rather than erroring
    X[col] = le.fit_transform(X[col].astype(str))

# Numeric nulls are intentionally KEPT — XGBoost handles them natively.
X.dtypes.value_counts()

## 4. Stratified train/test split

We hold out 20% for testing. **Stratified** splitting preserves the ~8% default rate in both train and test sets — important for imbalanced data, since a random split could otherwise end up with too few positives in one side.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,        # preserve class balance in both splits
    random_state=42,   # reproducibility
)

print(f"Train: {X_train.shape}, default rate {y_train.mean():.1%}")
print(f"Test:  {X_test.shape}, default rate {y_test.mean():.1%}")

## 5. Handle class imbalance with scale_pos_weight

The data is ~92% non-default / ~8% default. Left unaddressed, the model would be biased toward always predicting "non-default" (which is 92% accurate but useless).

XGBoost's `scale_pos_weight` parameter up-weights the minority (positive) class. The standard value is the ratio of negatives to positives, here roughly **11.4**, which tells the model to treat each default as ~11x more important than each non-default during training.

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"negatives={neg}, positives={pos}, scale_pos_weight={scale_pos_weight:.2f}")

## 6. Train the XGBoost model

Key parameters:
- `objective="binary:logistic"` — binary classification, outputs a probability
- `eval_metric="auc"` — optimize/track ROC-AUC during training
- `scale_pos_weight` — the imbalance correction from above
- `max_depth`, `learning_rate`, `n_estimators` — standard, conservative starting values
- early stopping on the test set to avoid overfitting and pick a good number of trees

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=30,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)

## 7. Evaluate

**ROC-AUC** is the primary metric — it measures how well the model ranks defaulters above non-defaulters across all thresholds and is robust to class imbalance (unlike accuracy). We also show a classification report and confusion matrix for a fuller picture of precision/recall on the minority class.

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Predicted probabilities for the positive class
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC: {auc:.4f}\n")
print("Classification report:")
print(classification_report(y_test, y_pred, digits=3))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

## 8. ROC curve and feature importance

The ROC curve visualizes the trade-off between true-positive and false-positive rates. Feature importance shows which features the model relied on most, a good sanity check that our engineered features (the bureau aggregates and ratios) are actually contributing.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, y_proba)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"XGBoost (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Home Credit Default Prediction")
plt.legend()
plt.show()

In [ ]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=X.columns)
top20 = importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(8, 6))
top20[::-1].plot(kind="barh")
plt.title("Top 20 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

top20

## 9. Summary

- Read the engineered feature table (application_train + aggregated bureau + ratio features) from HDFS.
- Label-encoded categoricals; kept numeric nulls for XGBoost to handle natively.
- Stratified 80/20 split to preserve the ~8% default rate.
- Trained XGBoost with `scale_pos_weight` to correct the class imbalance.
- Evaluated with ROC-AUC (the imbalance-appropriate metric) plus precision/recall and a confusion matrix.